# Zarr Stores with Xarray

## Learning Objectives:

- Learn how to read a Zarr store with a group hierarchical structure with `xr.DataTree` and `xarray`'s `"zarr"` engine
- Learn how to select and manipulate underlying `dask` arrays from a Zarr store
- Explore how to use Zarr stores with `xarray` for data analysis and visualization

## What is Zarr?

The Zarr data format is an open, community-maintained format designed for efficient, scalable storage of large N-dimensional arrays. It stores data as compressed and chunked arrays in a format well-suited to parallel processing and cloud-native workflows. 

### Zarr Data Organization:
- **Arrays**: N-dimensional arrays that can be chunked and compressed.
- **Groups**: A container for organizing multiple arrays and other groups with a hierarchical structure.
- **Metadata**: JSON-like metadata describing the arrays and groups, including information about data types, dimensions, chunking, compression, and user-defined key-value fields. 
- **Dimensions and Shape**: Arrays can have any number of dimensions, and their shape is defined by the number of elements in each dimension.
- **Coordinates & Indexing**: Zarr supports coordinate arrays for each dimension, allowing for efficient indexing and slicing.

The diagram below from [the Zarr v3 specification](https://wiki.earthdata.nasa.gov/display/ESO/Zarr+Format) showing the structure of a Zarr store:

![ZarrSpec](https://zarr-specs.readthedocs.io/en/latest/_images/terminology-hierarchy.excalidraw.png)


NetCDF and Zarr share similar terminology and functionality, but the key difference is that NetCDF is a single file, while Zarr is a directory-based “store” composed of many chunked files, making it better suited for distributed and cloud-based workflows.

## Opening a dataset with `xr.open_datatree(engine='zarr')`

Let's open up a precipitation zarr store. This dataset was derived from "GPM_3IMERGHH_07" and "M2T1NXFLX_5.12.4" products and has a group hierarchical structure.
**NOTE** We selected `consolidated=True`, Zarr supports consolidated metadata, which allows you to store all metadata in a single file and can improve performance when reading metadata.

In [ ]:
import xarray as xr

precipitation_store = "https://pub-45a1d62ac8d94c4c89f4dc63681a98ed.r2.dev/precipitation.zarr"

precip_dt = xr.open_datatree(precipitation_store, engine="zarr", chunks={}, consolidated=True)

### Accessing data variables
As with on disk storage methods, data from zarr stores in the cloud can be accessed with either dict-like syntax or method based syntax with the `"zarr"` engine.

In [ ]:
precip_dt.observed["precipitation"]

This returns an `xr.DataArray` object with the underly data as a chunked `dask.Array`.

## Time slicing

Like a dataset store on disk we can do time slicing on our zarr store. Each time slice is one hour data with a total of 10 hours of data. 

Let's try getting the first 5 hours of data with `.sel(time=)`. 

Since the time slices are ordered we can get a subset of the array of our time coordinate and pass it to the `.sel` method.

In [ ]:
time_index = precip_dt.time[0:5]
precip_dt.sel(time=time_index)

We can also subset by time with a `datetime` string.

In [ ]:
precip_dt.sel(time=slice("2021-08-29T07:30:00", "2021-08-29T16:30:00"))

Or by the index of our time dimension `.isel(time=slice())`

In [ ]:
precip_dt.isel(time=slice(0, 5))

## Chunking
Chunking is the process of dividing arrays into smaller pieces, which allows for parallel processing and efficient storage.

To examine the chunks in our Zarr store, with `xarray` you can use the `chunks` attribute on the `xr.DataArray` object.

In [ ]:
precip_dt.observed["precipitation"].data.chunks

### Selecting by chunks

Since our underlying data arrays are `dask.Array`, we can access data from each chunked array in our Zarr store. 

Let's get the first chunk of the "observed/precipitation" variable in our zarr store.

We add `.data` to our `xr.DataArray` to get access the `dask.Array`. The `.blocks[]` method allows you to index by chunk and `.compute()` returns the `np.ndarray`. 

In [ ]:
precip_dt.observed["precipitation"].data.blocks[0, 0, 0].compute()

In [ ]:
precip_dt.observed["precipitation"].mean(dim="time")

## Exercise

::::{admonition} Exercise
:class: tip

Can you calculate and plot the mean precipitation starting at 09:55 for the reanalysis group in this zarr store?

:::{admonition} Solution
:class: dropdown

```python
precip_dt.reanalysis['precipitation'].sel(time=slice('2021-08-29T09:55:00', '2021-08-29T16:30:00')).mean(dim='time').plot()
```
:::
::::